In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [55]:
test_folder_path = Path("./tests")
test = "demo"

In [56]:
with open(test_folder_path / test / f"{test}_specs.json" ) as f:
    test_specs = json.load(f)

norms_path = test_folder_path / test / f"{test}_norms.csv"
norms = pd.read_csv(norms_path)

In [221]:
def convert_to_matrices(test_specs):
    # get test length
    test_length = test_specs["length"]
    # init matrices to be returned
    df_straight = pd.DataFrame()
    df_reversed = pd.DataFrame()
    # iterate over scales
    for scale in test_specs["scales"]:
        # get scale label, straight and reversed items
        scale_label, straight_items_indices, reversed_items_indices = scale
        # iterate over type of items (either straight or reversed)
        for df, items_indices in [(df_straight, straight_items_indices), (df_reversed, reversed_items_indices)]:
            # init new series with zeroes 
            zeros = pd.Series(np.zeros(test_length))
            # correct items indices, since items are 1-based while matrix indices are 0-based
            matrix_indices = pd.Series(items_indices).sub(1)
            # "switch to 1" scale items in zeros series
            zeros[matrix_indices] = 1
            # add scale items matrix to df
            df[scale_label] = zeros
    return df_straight, df_reversed

def compute_raw_scores(data, test_specs, score_strategy = "sum", split_results = False):
    
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    
    ############################################################
    # straight items
    ############################################################
    
    # compute sum of straight items
    score_straight_sum = np.dot(answers.fillna(0), df_straight)
    # compute mean of straight items
    with np.errstate(divide='ignore', invalid='ignore'):
        score_straight_mean = np.true_divide(score_straight_sum, df_straight.sum(axis=0).to_numpy())
        score_straight_mean[score_straight_mean == np.inf] = 0
        score_straight_mean = np.nan_to_num(score_straight_mean)

    ############################################################
    # reversed items
    ############################################################
    
    # compute amount to use for reversed items
    rev = test_specs["likert"]["min"] + test_specs["likert"]["max"]
    # compute sum of reversed items
    score_reversed_sum = np.dot(rev - answers.fillna(rev), df_reversed)
    # compute mean of reversed items
    with np.errstate(divide='ignore', invalid='ignore'):
        score_reversed_mean = np.true_divide(score_reversed_sum, df_reversed.sum(axis=0).to_numpy())
        score_reversed_mean[score_reversed_mean == np.inf] = 0
        score_reversed_mean = np.nan_to_num(score_reversed_mean)

    #############################################################
    # final results
    ############################################################

    # in case we want split results for straight and reversed items
    if split_results:
        return score_straight_sum, score_reversed_sum
     # in case we want combined results
    else:
        # compute final results
        results.loc[:, df_straight.columns] = score_straight_sum + score_reversed_sum
        if score_strategy == "mean":
            results.loc[:, df_straight.columns] = score_straight_mean + score_reversed_mean
        
        # return results
        return results.astype(int)

def count_items_per_scale(data, test_specs, split_results = False):
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    # merge straight and reversed items matrices
    df_tot = df_straight + df_reversed
    # if we want results splitted by straight/reverse
    if split_results:
        return df_straight.sum(), df_reversed.sum()
    # otherwise return combined results
    return df_tot.sum()

def count_missing_items_per_scale(data, test_specs, split_results = False):
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    # merge straight and reversed items matrices
    df_tot = df_straight + df_reversed
    # if we want results splitted by straight/reverse
    if split_results:
        return (
            df_straight.apply(lambda x: answers.T.loc[x.astype(bool).values].isna().sum()),
            df_reversed.apply(lambda x: answers.T.loc[x.astype(bool).values].isna().sum())
        )
    # otherwise return combined results
    return df_tot.apply(lambda x: answers.T.loc[x.astype(bool).values].isna().sum())

def compute_raw_scores_compensate_for_missing_items(data, test_specs):
    
    # get splitted data that is needed for computing raw scores while compensating for missing items
    score_straight, score_reversed = compute_raw_scores(data, test_specs, score_strategy = "sum", split_results = True)
    missing_straight, missing_reversed = count_missing_items_per_scale(data, test_specs, split_results = True)
    items_straight, items_reversed = count_items_per_scale(data, test_specs, split_results = True)

    # init list
    components = []
    
    # compute corrected raw scores 
    for raw_scores, missing_items_by_scale, items_by_scale in [
        (score_straight, missing_straight, items_straight),
        (score_reversed, missing_reversed, items_reversed)
    ]: 
        # compute how many items where effectively responded (by scale)
        items_with_answers_by_scale = items_by_scale - missing_items_by_scale
        with np.errstate(divide='ignore', invalid='ignore'):
            # compute mean responses (by scale)
            mean_results= np.true_divide(raw_scores, items_with_answers_by_scale.astype(int))
            mean_results[mean_results == np.inf] = 0
            mean_results = np.nan_to_num(mean_results)
            # compute corrected results (by scale)
            corrected_results = mean_results * items_by_scale.to_numpy().T
            components.append(corrected_results)
    # return results
    return pd.DataFrame(components[0] + components[1], index=data.index, columns=items_straight.index)
    
def compute_standard_score(s, **kwargs):
    # get kwargs
    norms, type = kwargs["norms"],  kwargs["type"]
    # return standard scores
    return norms[norms["scale"].eq(s.name)].take(s)[type].values

In [222]:
# load data
data = pd.read_csv("data_demo.csv")
data.head()

,i1,i2,i3,i4,i5,i6,i7,i8,i9,10
0,1.0,1,1,1,1,1.0,1,1,1,1
1,2.0,2,2,2,2,2.0,2,2,2,2
2,3.0,3,3,3,3,3.0,3,3,3,3
3,4.0,4,4,4,4,4.0,4,4,4,4
4,5.0,5,5,5,5,5.0,5,5,5,5


In [223]:
items_by_scale = count_items_per_scale(data, test_specs)
items_by_scale

s1     5.0
s2     5.0
s3    10.0
dtype: float64

In [224]:
missing_items_by_scale = count_missing_items_per_scale(data, test_specs)
missing_items_by_scale

,s1,s2,s3
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
5,1,0,1
6,0,1,1
7,1,1,2


In [225]:
raw_scores = compute_raw_scores(data, test_specs)
raw_scores

,s1,s2,s3
0,5,25,30
1,10,20,30
2,15,15,30
3,20,10,30
4,25,5,30
5,4,25,29
6,10,16,26
7,12,12,24


In [226]:
raw_scores_corrected = compute_raw_scores_compensate_for_missing_items(data, test_specs)
raw_scores_corrected

,s1,s2,s3
0,5.0,25.0,30.0
1,10.0,20.0,30.0
2,15.0,15.0,30.0
3,20.0,10.0,30.0
4,25.0,5.0,30.0
5,5.0,25.0,30.0
6,10.0,20.0,30.0
7,15.0,15.0,30.0


In [227]:
t_scores = raw_scores.apply(compute_standard_score, norms=norms, type="tscore")
t_scores

,s1,s2,s3
0,41,98,54
1,60,77,54
2,65,65,54
3,77,60,54
4,98,41,54
5,41,98,52
6,60,68,41
7,65,65,41
